[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/glass_cyclide/glass_cyclide.ipynb)

# A Glass Dupin Cyclide

This example ray-traces a glass Dupin cyclide standing on a checkered floor in front of an opaque red ball as an orbiting camera circles the scene. Inverting a torus in an off-centre sphere swells one side of the ring and narrows the other into a Dupin cyclide, while subtracting a ball cuts a concave lens bite into its thick side where light refracts, reflects, and attenuates through the tinted glass.

In 3D conformal geometric algebra, each spatial point is modelled as a null antivector placed from the origin by a translator (`translator(offset) >> origin`), and a quadric is a symmetric polarity map sending a point to its polar sphere. Pairing that sphere with the point (`quadric(point) & point`) vanishes on the surface and turns negative inside, while on the surface `quadric(point)` is tangent to the shape so flattening it against `horizon` yields the outward normal. Because a torus is a conformal quadric and sphere inversion is an odd versor, sandwiching the torus map (`inversion >> torus(inversion << Point)`) produces the Dupin cyclide directly, and negating a ball's quadric carves the spherical bite out of the glass in a WGSL shader written in numga.

In Cartesian coordinate notation, a torus is the quartic polynomial $(x^2 + y^2 + z^2 + R^2 - r^2)^2 - 4 R^2 (x^2 + y^2) = 0$, and inverting $\mathbf{x} = (x, y, z)$ in a sphere of centre $\mathbf{c}$ and radius $a$ substitutes the rational map $\mathbf{x}' = \mathbf{c} + a^2 (\mathbf{x} - \mathbf{c}) / \|\mathbf{x} - \mathbf{c}\|^2$ and clears denominators to form a separate quartic cyclide polynomial $f(\mathbf{x}) = 0$. Placing the solid by a rotation matrix $Q \in \mathrm{SO}(3)$ and translation $\mathbf{t}$ then evaluates $f(Q^\top(\mathbf{x} - \mathbf{t}))$ along each ray $\mathbf{o} + s\,\mathbf{d}$ and differentiates $Q\,\nabla f(Q^\top(\mathbf{x} - \mathbf{t}))$ separately for the surface normal.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import CGA3D
from numga.wgsl import Shader, supersampled
from examples.util.animation import save_figure, show_animation
from examples.util.panels import image_figure
from examples.quadrics.glass_cyclide import render

ga = CGA3D
mv = NumpyContext(ga).multivector
# Euclidean subspace spanned by x, y, and z.
space = Algebra("x+y+z+")
# Spheres and planes are vectors, while planes through the origin lie within x, y, and z.
# Conformal points are null antivectors, flat directions are dual to origin planes, and motors move space conformally.
Sphere = ga.gatype.vector()                                                    # Vector+
OriginPlane = Sphere.derive.within(space)                                      # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
Direction = OriginPlane.derive.dual                                            # Antivector-
Motor = ga.gatype.rotor()                                                      # Rotor+
Rotation = Motor.derive.within(space)                                          # Rotor+
Quadric = ga.gatype((Sphere, Point))                                           # Sphere+ <- Point-
# Conformal origin, point at infinity, and horizon plane whose pairing with a point reads its weight.
origin = (mv.w.dual() + mv.e.dual()) / 2                                       # [] Point-
infinity = mv.e.dual() - mv.w.dual()                                           # [] Point-
horizon = mv.e - mv.w                                                          # [] Sphere+

## Inverting a torus into a Dupin cyclide

Inverting a torus in a sphere displaced from its centre swells the far lobe of the ring and pinches the near lobe into a Dupin cyclide. Because sphere inversion is a conformal versor, applying the same sandwich to the torus quadric and to points sampled on the torus keeps those inverted points on the cyclide surface after dividing out the square of each point's conformal weight `horizon & point`. At every surface point, evaluating the quadric returns the tangent sphere there, and cancelling its conformal curvature term against `horizon` flattens that sphere into the outward normal direction.

In [ ]:
torus_radius, torus_tube = 2.5, 0.9
inversion_offset, inversion_radius = 8.0, 6.25
n_around, n_across = 24, 12

In [ ]:
def translator(offset: Direction) -> Motor:
    """Exponentiating the dual of `infinity & offset` gives the conformal motor (`[...] Motor+`) translating along `offset`."""
    return (-(infinity & offset).dual() / 2).exp()


def sphere(centre: Point, radius: float) -> Sphere:
    """Subtracting half `radius**2 * horizon` from `centre`'s zero-radius sphere forms the sphere (`[] Sphere+`) positive inside."""
    return centre.dual_inverse() - radius**2 / 2 * horizon


inversion = (translator(mv.x.dual() * inversion_offset) >> sphere(origin, inversion_radius)).normalized()  # [] Sphere+

In [ ]:
def torus(radius: float, tube: float) -> Quadric:
    """The core sphere, the z plane, and `horizon` combine into the solid torus quadric (`[] Sphere+ <- Point-`) of `radius` and `tube`."""
    core_sphere = sphere(origin, np.hypot(radius, tube))                                             # [] Sphere+
    return core_sphere * (core_sphere & Point) + radius**2 * (mv.z * (mv.z & Point) - tube**2 * horizon * (horizon & Point))


def torus_points(radius: float, tube: float, n_around: int, n_across: int) -> Point:
    """Tube and ring rotations composed with radial translators carry `origin` across a torus surface (`[n_across, n_around] Point-`)."""
    ring_turns = (mv.xy * (-np.linspace(0, 2 * np.pi, n_around, endpoint=False) / 2)).exp()          # [n_around] Rotation+
    tube_turns = (mv.xz * (-np.linspace(0, 2 * np.pi, n_across, endpoint=False)[:, None] / 2)).exp()  # [n_across, 1] Rotation+
    motors = ring_turns * translator(mv.x.dual() * radius) * tube_turns * translator(mv.x.dual() * tube)  # [n_across, n_around] Motor+
    return motors >> origin                                                                          # [n_across, n_around] Point-


cyclide = inversion >> torus(torus_radius, torus_tube)(inversion << Point)                           # [] Sphere+ <- Point-
cyclide_points = inversion >> torus_points(torus_radius, torus_tube, n_around, n_across)             # [n_across, n_around] Point-
surface_residuals = (cyclide(cyclide_points) & cyclide_points) / (horizon & cyclide_points).squared()  # [n_across, n_around] Scalar+

In [ ]:
def outward_normal(tangent: Sphere, point: Point) -> Direction:
    """Flattening a tangent sphere against `horizon` at `point` extracts the surface's outward normal direction (`[...] Direction-`)."""
    # Adding the point's zero-radius sphere weighted by tangent | horizon cancels the conformal curvature term at point,
    # leaving the tangent plane whose Euclidean part dualizes to the outward normal direction.
    return (tangent + (tangent | horizon) * point.dual_inverse()).cast(OriginPlane).dual()


normals = outward_normal(cyclide(cyclide_points), cyclide_points)                                    # [n_across, n_around] Direction-
exterior_points = translator((normals / normals.antinorm()) * 0.01) >> cyclide_points                # [n_across, n_around] Point-
exterior_forms = (cyclide(exterior_points) & exterior_points) / (horizon & exterior_points).squared()  # [n_across, n_around] Scalar+

## Composing the scene solids

A conformal motor centres the cyclide, tilts it upright, and lifts it above the floor, while a ball is constructed by symmetrizing a sphere or plane with `horizon` so the quadratic form turns negative on the interior. Intersecting the placed cyclide with a negated ball subtracts a concave spherical bite from its swollen side, and pairing that glass ring with an opaque red ball behind it and a planar half-space below forms the complete scene.

In [ ]:
cyclide_centre, cyclide_radius, cyclide_tilt = mv.x.dual() * 2.04, 2.6, np.radians(55.0)
ring_lift = mv.z.dual() * 2.25                                                                 # [] Direction-
bite_centre = translator(mv.x.dual() * 0.6 + mv.y.dual() * 1.2 + mv.z.dual() * 0.9) >> origin  # [] Point-
bite_radius = 0.9
ball_centre = translator(mv.x.dual() * 0.6 + mv.y.dual() * 3.4 + mv.z.dual() * 1.1) >> origin  # [] Point-
ball_radius = 1.1
floor_radius, tile_density = 60.0, 0.8
colours = np.array([[0.9, 0.96, 0.97], [0.85, 0.22, 0.16], [0.88, 0.87, 0.85]])
transparencies = np.array([0.96, 0.0, 0.0])
refractions = np.array([1.5, 1.0, 1.0])

In [ ]:
def ball(surface: Sphere) -> Quadric:
    """Symmetrizing a sphere or plane with `horizon` forms the quadric (`[] Sphere+ <- Point-`) negative on its interior."""
    return (surface * (horizon & Point) + horizon * (surface & Point)) / 2


centring = translator(-cyclide_centre)                                                         # [] Motor+
tilt = (mv.yz * (-cyclide_tilt / 2)).exp()                                                     # [] Rotation+
ring_motor = translator(ring_lift) * tilt * centring                                           # [] Motor+
shapes = [
    # Intersecting the placed cyclide with the negated bite ball carves a concave lens out of the thick side.
    [ring_motor >> cyclide(ring_motor << Point), -ball(ring_motor >> sphere(bite_centre, bite_radius))],
    [ball(sphere(ball_centre, ball_radius))],
    [ball(-mv.z)],
]
counts = np.array([len(shape) for shape in shapes])
ends = np.cumsum(counts)
quadrics = stack([quadric.cast(Quadric) for shape in shapes for quadric in shape])             # [4] Sphere+ <- Point-
centres = stack([translator(ring_lift) >> origin, ball_centre, origin])                        # [3] Point-
tile_x = stack([0 * mv.x, 0 * mv.x, mv.x * tile_density])                                      # [3] OriginPlane+
tile_y = stack([0 * mv.y, 0 * mv.y, mv.y * tile_density])                                      # [3] OriginPlane+
solid_table = dict(
    first=ends - counts,
    end=ends,
    centre=centres,
    radius=np.array([cyclide_radius, ball_radius * 1.01, floor_radius]),
    colour=colours,
    transparency=transparencies,
    refraction=refractions,
    tile_x=tile_x,
    tile_y=tile_y,
)

## Ray tracing through glass

Each pixel traces rays through the conformal quadric solids in WGSL using `supersampled()`, which averages a rotated subpixel grid per pixel on a GPU. Inside each solid's bounding sphere, stepping brackets the first sign change of the largest quadric pairing `quadric(point) & point` and bisection refines the hit distance, after which the tangent sphere supplies the interface normal for Snell refraction, Schlick-Fresnel reflection, total internal reflection, and exponential tint attenuation.

In [ ]:
source = supersampled() + """
// Each solid is the intersection of quadrics[first .. end] inside a bounding sphere around centre,
// carrying optical material parameters and two origin planes that lay a unit checker pattern across it.
struct Solid {
    first: u32,
    end: u32,
    centre: Point,
    radius: f32,
    colour: vec3<f32>,
    transparency: f32,
    refraction: f32,
    tile_x: OriginPlane,
    tile_y: OriginPlane,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> turn: Rotation;
var<uniform> eye: Point;
var<uniform> lamp: Direction;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

const far = 1e6;
const clearance = 1e-3;
const n_steps = 40u;
const n_halvings = 18u;
const n_bounces = 8u;
const ambient = 0.3;
const glass_shadow = 0.75;
const checker_contrast = 0.18;
const horizon_colour = vec3<f32>(0.86, 0.88, 0.92);
const zenith_colour = vec3<f32>(0.36, 0.5, 0.72);
const sun_brightness = 6.0;
const sun_sharpness = 2000.0;

// Sphere around centre whose pairing with an interior point is positive.
fn sphere(centre: Point, radius: f32) -> Sphere {
    return centre.dual_inverse() - radius * radius / 2 * horizon;
}

// Rate of change of a conformal point per unit step along a unit direction, given by its commutator with the translator generator.
fn velocity(start: Point, direction: Direction) -> Point {
    return start.commutator((infinity & direction).dual());
}

// Point at a given distance along a ray from a unit-weight start, expanded so single-precision start weight never multiplies distance squared.
fn ray_point(start: Point, direction: Direction, distance: f32) -> Point {
    return start + velocity(start, direction) * distance + infinity * (distance * distance / 2);
}

// Largest quadric form of the solid at point, negative strictly inside all of the solid's quadrics.
fn signed_outside(solid: Solid, point: Point) -> f32 {
    var largest = -far;
    for (var index = solid.first; index < solid.end; index++) {
        largest = max(largest, quadrics[index](point) & point);
    }
    return largest;
}

// First positive ray distance where the solid's surface is crossed, or far when the ray misses.
fn crossing(solid: Solid, origin: Point, direction: Direction) -> f32 {
    // Pairing the bounding sphere with the ray start and its velocity gives the chord midpoint and half-chord length squared.
    let bounding = sphere(solid.centre, solid.radius);
    let axial_distance = bounding & velocity(origin, direction);
    let half_chord_squared = axial_distance * axial_distance + 2 * (bounding & origin);
    let start = max(axial_distance - sqrt(max(half_chord_squared, 0.0)), clearance);
    let end = axial_distance + sqrt(max(half_chord_squared, 0.0));
    if (half_chord_squared <= 0 || end <= start) {
        return far;
    }
    let started_outside = signed_outside(solid, ray_point(origin, direction, start)) > 0;
    var before = start;
    for (var step = 1u; step <= n_steps; step++) {
        let after = start + (end - start) * f32(step) / f32(n_steps);
        if ((signed_outside(solid, ray_point(origin, direction, after)) > 0) != started_outside) {
            var low = before;
            var high = after;
            for (var halving = 0u; halving < n_halvings; halving++) {
                let middle = (low + high) / 2;
                if ((signed_outside(solid, ray_point(origin, direction, middle)) > 0) == started_outside) {
                    low = middle;
                } else {
                    high = middle;
                }
            }
            return high;
        }
        before = after;
    }
    return far;
}

struct Hit {
    distance: f32,
    solid: u32,
}

fn nearest(origin: Point, direction: Direction) -> Hit {
    var hit = Hit(far, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let distance = crossing(solids[index], origin, direction);
        if (distance < hit.distance) {
            hit = Hit(distance, index);
        }
    }
    return hit;
}

// Unit origin plane parallel to the solid's tangent plane at point, flattened from the active quadric's tangent sphere.
fn surface_plane(solid: Solid, point: Point) -> OriginPlane {
    var active = solid.first;
    for (var index = solid.first; index < solid.end; index++) {
        if ((quadrics[index](point) & point) > (quadrics[active](point) & point)) {
            active = index;
        }
    }
    let tangent = quadrics[active](point);
    return (tangent + (tangent | horizon) * point.dual_inverse()).cast(OriginPlane).normalized();
}

// Sky radiance from vertical elevation above the floor plane and alignment with the solar lamp direction.
fn sky(direction: Direction) -> vec3<f32> {
    let elevation = zenith_plane & direction;
    let sun_alignment = lamp.dual_inverse() & direction;
    return mix(horizon_colour, zenith_colour, clamp(elevation, 0.0, 1.0)) + sun_brightness * pow(max(sun_alignment, 0.0), sun_sharpness);
}

// Direct lamp transmittance reaching point, attenuated partially by glass solids and blocked completely by opaque ones.
fn sunlight(point: Point) -> f32 {
    var light = 1.0;
    for (var index = 0u; index < arrayLength(&solids); index++) {
        if (crossing(solids[index], point, lamp) < far) {
            light *= solids[index].transparency * glass_shadow;
        }
    }
    return light;
}

// Opaque diffuse shading with lamp shadow and planar checker modulation.
fn matte_colour(solid: Solid, point: Point, facing: OriginPlane) -> vec3<f32> {
    let lit = max(facing & lamp, 0.0) * sunlight(ray_point(point, facing.dual(), clearance));
    let parity = abs((floor(solid.tile_x & point) + floor(solid.tile_y & point)) % 2.0);
    return solid.colour * (1 - checker_contrast * parity) * (ambient + (1 - ambient) * lit);
}

// Single-bounce reflection lookup returning either the sky or an unrefracted surface glimpse.
fn glimpse(origin: Point, direction: Direction) -> vec3<f32> {
    let hit = nearest(origin, direction);
    if (hit.distance >= far) {
        return sky(direction);
    }
    let solid = solids[hit.solid];
    let point = ray_point(origin, direction, hit.distance);
    let facing = surface_plane(solid, point);
    return mix(matte_colour(solid, point, facing), sky(direction), solid.transparency);
}

// Radiance along the camera ray through a normalized image sample location.
fn sample(place: vec2<f32>) -> Sample {
    let screen_offset = screen(place) * half_view;
    let ray_vector = turn >> (view_axis + (screen_axes * screen_offset).sum());
    var direction = ray_vector / ray_vector.antinorm();
    var position = eye;
    var colour = vec3<f32>(0.0);
    var throughput = vec3<f32>(1.0);
    for (var bounce = 0u; bounce < n_bounces; bounce++) {
        let hit = nearest(position, direction);
        if (hit.distance >= far) {
            colour += throughput * sky(direction);
            break;
        }
        let solid = solids[hit.solid];
        position = ray_point(position, direction, hit.distance);
        let incident_normal = surface_plane(solid, position);
        let entering = (incident_normal & direction) < 0;
        // Rays exiting tinted glass attenuate exponentially per unit path length inside the solid.
        throughput *= select(pow(solid.colour, vec3<f32>(hit.distance)), vec3<f32>(1.0), entering || solid.transparency == 0);
        let facing = select(-incident_normal, incident_normal, entering);
        colour += throughput * (1 - solid.transparency) * matte_colour(solid, position, facing);
        // Snell refraction and plane reflection across the interface, falling back to total internal reflection past the critical angle.
        let cosine = -(facing & direction);
        let ratio = select(solid.refraction, 1 / solid.refraction, entering);
        let sine_squared = ratio * ratio * (1 - cosine * cosine);
        let reflected_ray = facing >> direction;
        let refracted_ray = direction * ratio + facing.dual() * (ratio * cosine - sqrt(max(1 - sine_squared, 0.0)));
        // Schlick approximation to the Fresnel reflectance at dielectric interfaces.
        let normal_reflectance = ((solid.refraction - 1) / (solid.refraction + 1)) * ((solid.refraction - 1) / (solid.refraction + 1));
        let reflectance = normal_reflectance + (1 - normal_reflectance) * pow(1 - cosine, 5.0);
        let total_reflection = sine_squared > 1;
        colour += throughput * solid.transparency * select(reflectance, 0.0, total_reflection) * glimpse(ray_point(position, reflected_ray, clearance), reflected_ray);
        throughput *= solid.transparency * select(1 - reflectance, 1.0, total_reflection);
        direction = select(refracted_ray, reflected_ray, total_reflection);
        position = ray_point(position, direction, clearance);
        if (max(throughput.x, max(throughput.y, throughput.z)) < 0.01) {
            break;
        }
    }
    return colour;
}
"""

screen_axes = stack((mv.x.dual(), mv.y.dual()))                                # [2] Direction-
view_axis = -mv.z.dual()                                                       # [] Direction-
zenith_plane = mv.z                                                            # [] OriginPlane+
tracer = Shader(
    source,
    ga,
    dict(Sphere=Sphere, OriginPlane=OriginPlane, Point=Point, Direction=Direction, Rotation=Rotation, Quadric=Quadric),
    dict(infinity=infinity, horizon=horizon, screen_axes=screen_axes, view_axis=view_axis, zenith_plane=zenith_plane),
)

In [ ]:
lamp_elevation, lamp_azimuth = np.radians(49.0), np.radians(-50.0)
target = translator(mv.z.dual() * 1.9) >> origin                               # [] Point-
distance, elevation, half_view = 11.0, np.radians(20.0), 0.31
image_shape = (450, 720)

In [ ]:
def orbit_cameras(
    target: Point, distance: float, elevation: float, azimuths: np.ndarray,
) -> tuple[Rotation, Point]:
    """Azimuth and elevation rotations place orbiting camera poses (`[...] Rotation+`, `[...] Point-`) at `distance` around `target`."""
    pitch = (mv.yz * (-(np.pi / 2 - elevation) / 2)).exp()                                           # [] Rotation+
    turns = (mv.xy * (-azimuths / 2)).exp() * pitch                                                  # [...] Rotation+
    offsets = (turns >> mv.z.dual()) * distance                                                      # [...] Direction-
    eyes = translator(offsets) >> target                                                             # [...] Point-
    return turns, eyes


lamp_turn = (mv.xy * (-lamp_azimuth / 2)).exp() * (mv.xz * (-lamp_elevation / 2)).exp()              # [] Rotation+
lamp = lamp_turn >> mv.x.dual()                                                                      # [] Direction-
still_turns, still_eyes = orbit_cameras(target, distance, elevation, np.zeros(1))                    # [1] Rotation+, Point-
still_image, = render.camera_frames(
    tracer, quadrics, solid_table, still_turns, still_eyes, lamp, half_view, image_shape,
)
save_figure(image_figure(still_image), "glass_cyclide");

Through the swollen right side of the glass cyclide the checkered floor tiles magnify and fold back on themselves, whereas through the narrow left side they shift only slightly. The spherical bite cut into the thick lobe acts as a concave diverging lens with a bright grazing rim, and the opaque red ball behind the ring appears both directly above the opening and refracted upside down through the lower glass wall.

## Orbiting the glass cyclide

Circling the camera around the vertical axis sweeps every cross-section of the cyclide across the red ball and the checkered floor. Tracing all 96 supersampled frames takes several seconds.

In [ ]:
animation_shape, n_frames, frame_duration_ms = (330, 528), 96, 40
azimuths = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
orbit_turns, orbit_eyes = orbit_cameras(target, distance, elevation, azimuths)                       # [n_frames] Rotation+, Point-
show_animation(
    render.camera_frames(tracer, quadrics, solid_table, orbit_turns, orbit_eyes, lamp, half_view, animation_shape),
    "glass_cyclide",
    frame_duration_ms,
)

As the camera orbits from edge-on to face-on, total internal reflection brightens the curved silhouette while the refracted image of the red ball splits into secondary arcs that travel around the tube and across the sharp rim of the concave bite.

In [ ]:
# checks
assert (surface_residuals.abs() < 1e-10).all()
assert (exterior_forms > 0).all()